# ROGII: Typewell GR Motif Ambiguity Atlas

Why can a GR-guided tracker jump to the wrong TVT branch even when its local curve match looks excellent? One reason is repeated texture inside the typewell itself.

This notebook audits all 773 paired typewell files for distant, shape-similar 40-ft GR motifs. Each window is centered and normalized, so the score focuses on shape rather than gain or offset. Candidate pairs must be at least 80 ft apart. We also inventory byte-identical typewell files, because treating duplicated references as independent evidence can distort validation.

The result is a structural ambiguity diagnostic, not a predictor. It uses no suffix labels, creates no submission, and spends no competition submission.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data was not found')
TRAIN_DIR = DATA_ROOT / 'train'
GRID_STEP_FT = 1.0
WINDOW_FT = 40
WINDOW_STEP_FT = 10
MIN_SEPARATION_FT = 80
paths = sorted(TRAIN_DIR.glob('*__typewell.csv'))
print('Data root:', DATA_ROOT)
print('Typewell files:', len(paths))

## Exact reference duplication

First we hash the raw typewell CSV bytes. Exact duplicates are not automatically a problem: several horizontal wells may legitimately share one reference. They are a validation-design fact, however. A random well split can place the same typewell curve on both sides, so claims about generalizing to unseen references should group by typewell hash.

In [ ]:
hash_rows = []
for path in paths:
    hash_rows.append({
        'well_id': path.name.removesuffix('__typewell.csv'),
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    })
hashes = pd.DataFrame(hash_rows)
groups = (
    hashes.groupby('sha256')['well_id']
    .agg(group_size='size', well_ids=lambda values: ', '.join(values))
    .sort_values('group_size', ascending=False)
)
print('Unique byte-level typewells:', len(groups))
print('Duplicate hash groups:', int((groups['group_size'] > 1).sum()))
print('Largest group:', int(groups['group_size'].max()))
display(groups[groups['group_size'] > 1].head(15))

## Distant motif search

For comparability, every clean typewell is interpolated to a 1-ft TVT grid. We extract a 40-ft window every 10 ft, subtract its mean, divide by its Euclidean norm, and take the dot product between standardized windows. This is normalized shape similarity in `[-1, 1]`.

Pairs separated by less than 80 ft are masked. That removes the trivial similarity of adjacent windows and asks a more operational question: does a distant location provide a convincing alternative GR explanation?

In [ ]:
def clean_grid(path):
    frame = pd.read_csv(path, usecols=['TVT', 'GR']).dropna()
    frame = frame.sort_values('TVT').drop_duplicates('TVT')
    grid = np.arange(np.ceil(frame['TVT'].min()), np.floor(frame['TVT'].max()) + GRID_STEP_FT, GRID_STEP_FT)
    gr = np.interp(grid, frame['TVT'].to_numpy(dtype=float), frame['GR'].to_numpy(dtype=float))
    return grid, gr

def motif_audit(path):
    grid, gr = clean_grid(path)
    window_rows = int(round(WINDOW_FT / GRID_STEP_FT))
    step_rows = int(round(WINDOW_STEP_FT / GRID_STEP_FT))
    starts = np.arange(0, len(gr) - window_rows + 1, step_rows, dtype=int)
    windows = np.stack([gr[start:start + window_rows] for start in starts])
    windows = windows - windows.mean(axis=1, keepdims=True)
    norms = np.linalg.norm(windows, axis=1)
    valid = norms > 1e-8
    windows = windows[valid] / norms[valid, None]
    starts = starts[valid]
    similarity = windows @ windows.T
    separation = np.abs(starts[:, None] - starts[None, :]) * GRID_STEP_FT
    similarity[separation < MIN_SEPARATION_FT] = -np.inf
    flat_index = int(np.argmax(similarity))
    first, second = np.unravel_index(flat_index, similarity.shape)
    per_window_best = similarity.max(axis=1)
    return {
        'well_id': path.name.removesuffix('__typewell.csv'),
        'grid_rows': len(grid),
        'tvt_span_ft': float(grid[-1] - grid[0]),
        'windows': len(starts),
        'best_distant_similarity': float(similarity[first, second]),
        'best_separation_ft': float(separation[first, second]),
        'first_start_tvt': float(grid[starts[first]]),
        'second_start_tvt': float(grid[starts[second]]),
        'fraction_windows_match_090': float(np.mean(per_window_best > 0.90)),
        'fraction_windows_match_095': float(np.mean(per_window_best > 0.95)),
    }

records = [motif_audit(path) for path in paths]
motifs = pd.DataFrame(records).merge(hashes, on='well_id', validate='one_to_one')
display(motifs.head())

In [ ]:
summary = pd.Series({
    'typewells': len(motifs),
    'median TVT span ft': motifs['tvt_span_ft'].median(),
    'median best distant similarity': motifs['best_distant_similarity'].median(),
    'p90 best distant similarity': motifs['best_distant_similarity'].quantile(0.90),
    'p95 best distant similarity': motifs['best_distant_similarity'].quantile(0.95),
    'fraction with best similarity > 0.90': (motifs['best_distant_similarity'] > 0.90).mean(),
    'fraction with best similarity > 0.95': (motifs['best_distant_similarity'] > 0.95).mean(),
})
display(summary.to_frame('value').style.format('{:.4f}'))
display(motifs.nlargest(15, 'best_distant_similarity')[[
    'well_id', 'tvt_span_ft', 'best_distant_similarity', 'best_separation_ft',
    'first_start_tvt', 'second_start_tvt', 'fraction_windows_match_090',
]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(motifs['best_distant_similarity'], bins=35, color='#2a6fbb', alpha=0.85)
axes[0].axvline(0.90, color='#c23b22', linestyle='--', label='0.90')
axes[0].axvline(0.95, color='#7b2d26', linestyle=':', label='0.95')
axes[0].set(title='Strongest distant 40-ft motif match', xlabel='normalized similarity', ylabel='typewells')
axes[0].legend()
scatter = axes[1].scatter(
    motifs['tvt_span_ft'], motifs['best_distant_similarity'],
    c=motifs['fraction_windows_match_090'], cmap='viridis', s=18, alpha=0.75,
)
axes[1].set(title='Ambiguity versus reference length', xlabel='typewell TVT span (ft)', ylabel='best distant similarity')
fig.colorbar(scatter, ax=axes[1], label='fraction of windows with >0.90 distant match')
plt.tight_layout()
plt.show()

## What a high-similarity alternative looks like

The left column overlays the two centered, normalized 40-ft motifs. The right column shows where they occur in the full typewell. A tracker that uses only short local GR shape can assign high likelihood to both branches; trajectory dynamics, longer context, geology, and an anchor prior must break the tie.

In [ ]:
examples = motifs.nlargest(3, 'best_distant_similarity')
fig, axes = plt.subplots(len(examples), 2, figsize=(13, 10))
for row_index, (_, record) in enumerate(examples.iterrows()):
    path = TRAIN_DIR / f'{record.well_id}__typewell.csv'
    grid, gr = clean_grid(path)
    length = int(round(WINDOW_FT / GRID_STEP_FT))
    starts = [
        int(round((record.first_start_tvt - grid[0]) / GRID_STEP_FT)),
        int(round((record.second_start_tvt - grid[0]) / GRID_STEP_FT)),
    ]
    for label, start in zip(['motif A', 'motif B'], starts):
        window = gr[start:start + length]
        standardized = (window - window.mean()) / max(np.linalg.norm(window - window.mean()), 1e-12)
        axes[row_index, 0].plot(np.arange(length) * GRID_STEP_FT, standardized, label=label)
        axes[row_index, 1].axvspan(grid[start], grid[start + length - 1], alpha=0.22)
    axes[row_index, 0].set(
        title=f'{record.well_id}: similarity={record.best_distant_similarity:.3f}',
        xlabel='offset within motif (ft)', ylabel='standardized GR',
    )
    axes[row_index, 0].legend()
    axes[row_index, 1].plot(grid, gr, color='#444444', linewidth=0.9)
    axes[row_index, 1].set(title='Full typewell locations', xlabel='TVT', ylabel='GR')
plt.tight_layout()
plt.show()

## How to use this diagnostic safely

- High motif similarity means a local observation may be non-identifying; it does not prove a specific tracker will fail.
- Low maximum similarity is not a confidence certificate. Horizontal GR calibration error, missing intervals, rate drift, and boundary effects still matter.
- Confidence gates must use only information observable at inference time. Suffix TVT error is valid for evaluation, never for selecting a test-well branch.
- If the validation claim is transfer to unseen reference curves, group folds by the exact typewell hash or a documented near-duplicate rule.
- Particle or HMM methods should preserve multiple branches longer when the typewell contains strong distant alternatives, then combine them with trajectory continuity and a conservative anchor prior.

A beautiful local GR match is evidence. It is not necessarily identification.